# 10c — Cross-Dataset Error Analysis

This notebook analyzes the harder transfer direction:
- train on BanglaSarc3-binary
- test on Ben-Sarc-binary

It compares:
- plain cross-dataset BanglaBERT
- cross-dataset BanglaBERT + FGM

In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer
from sklearn.metrics import confusion_matrix

TABLES = Path("../04_outputs/tables")
SPLITS = Path("../01_data/interim/splits")
CHECKPOINTS = Path("../03_models/checkpoints")
MODEL_NAME = "csebuetnlp/banglabert"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LENGTH,
    )

def resolve_checkpoint_dir(checkpoint_root):
    checkpoint_root = Path(checkpoint_root)
    if (checkpoint_root / "config.json").exists():
        return checkpoint_root
    ckpts = sorted(
        [p for p in checkpoint_root.glob("checkpoint-*") if p.is_dir()],
        key=lambda p: int(p.name.split("-")[-1])
    )
    if not ckpts:
        raise FileNotFoundError(f"No checkpoint-* folder found inside: {checkpoint_root}")
    trainer_state_file = checkpoint_root / "trainer_state.json"
    if trainer_state_file.exists():
        with open(trainer_state_file, "r", encoding="utf-8") as f:
            trainer_state = json.load(f)
        best_ckpt = trainer_state.get("best_model_checkpoint", None)
        if best_ckpt:
            best_ckpt = Path(best_ckpt)
            if best_ckpt.exists():
                return best_ckpt
    return ckpts[-1]

/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def load_binary_predictions(checkpoint_root, split_file):
    df = pd.read_csv(split_file).copy()
    ds_df = df[["text", "label_binary"]].rename(columns={"label_binary": "label"})
    ds = Dataset.from_pandas(ds_df, preserve_index=False)
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")

    checkpoint_dir = resolve_checkpoint_dir(checkpoint_root)
    print("Loading:", checkpoint_dir)

    model = AutoModelForSequenceClassification.from_pretrained(checkpoint_dir)
    trainer = Trainer(model=model)
    output = trainer.predict(ds)
    logits = output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = np.argmax(probs, axis=1)

    out = df.copy()
    out["y_true"] = df["label_binary"].astype(int).to_numpy()
    out["y_pred"] = preds
    out["prob_0"] = probs[:, 0]
    out["prob_1"] = probs[:, 1]
    out["correct"] = out["y_true"] == out["y_pred"]
    out["pred_conf"] = np.where(out["y_pred"] == 1, out["prob_1"], out["prob_0"])
    return out

In [3]:
def compare_models(base_df, improved_df, n=20):
    merged = pd.DataFrame({
        "text": base_df["text"],
        "y_true": base_df["y_true"],
        "base_pred": base_df["y_pred"],
        "base_correct": base_df["correct"],
        "base_conf": base_df["pred_conf"],
        "improved_pred": improved_df["y_pred"],
        "improved_correct": improved_df["correct"],
        "improved_conf": improved_df["pred_conf"],
    })
    improved_cases = merged[(merged["base_correct"] == False) & (merged["improved_correct"] == True)].copy()
    degraded_cases = merged[(merged["base_correct"] == True) & (merged["improved_correct"] == False)].copy()
    return improved_cases.head(n), degraded_cases.head(n)

def top_errors(df, n=20):
    return df[df["correct"] == False].sort_values("pred_conf", ascending=False)[
        ["text", "y_true", "y_pred", "prob_0", "prob_1", "pred_conf"]
    ].head(n)

In [4]:
cross_plain = load_binary_predictions(
    "../03_models/checkpoints/cross_banglasarc3_binary_to_ben_sarc_binary",
    "../01_data/interim/splits/ben_sarc_binary_test.csv",
)
cross_fgm = load_binary_predictions(
    "../03_models/checkpoints/cross_fgm_banglasarc3_binary_to_ben_sarc_binary",
    "../01_data/interim/splits/ben_sarc_binary_test.csv",
)

print("Cross plain confusion matrix:")
print(confusion_matrix(cross_plain["y_true"], cross_plain["y_pred"]))

print("Cross FGM confusion matrix:")
print(confusion_matrix(cross_fgm["y_true"], cross_fgm["y_pred"]))

Map: 100%|██████████| 2564/2564 [00:00<00:00, 13262.36 examples/s]


Loading: ../03_models/checkpoints/cross_banglasarc3_binary_to_ben_sarc_binary/checkpoint-1604


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9213.82it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Map: 100%|██████████| 2564/2564 [00:00<00:00, 25749.92 examples/s]


Loading: ../03_models/checkpoints/cross_fgm_banglasarc3_binary_to_ben_sarc_binary/checkpoint-1604


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 9182.71it/s]
/Users/sefayet/.pyenv/versions/3.11.6/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Cross plain confusion matrix:
[[839 443]
 [364 918]]
Cross FGM confusion matrix:
[[725 557]
 [317 965]]


In [5]:
print("Top cross-dataset errors — plain")
display(top_errors(cross_plain, n=20))

print("Top cross-dataset errors — FGM")
display(top_errors(cross_fgm, n=20))

Top cross-dataset errors — plain


,text,y_true,y_pred,prob_0,prob_1,pred_conf
2325,যেমন কর্ম তেমন ফল এখানে তার জন্য মায়া কান্দন ক...,1,0,0.976927,0.023073,0.976927
2205,ব্যবসায়ীরা যতদিন ব্যবসা ছেড়ে রাজনীতি আর সরকারি...,1,0,0.976172,0.023828,0.976172
1288,লেভেল আর মেন্টালিটি নিয়ে কথা বলা মানুষগুলাই ক্...,1,0,0.975576,0.024424,0.975576
809,সাংবাদিকদের এরকম পরিস্থিতি হওয়ার জন্য তাদের চা...,1,0,0.973890,0.026110,0.973890
1671,কিছু মানুষ যে খেলা না বুঝে এ চিল্লায় আপনি তার ...,1,0,0.973867,0.026133,0.973867
2181,সস্তা জনপ্রিয়তাকে কাজে লাগিয়ে টাউটামি যে শুরু ...,1,0,0.973808,0.026192,0.973808
991,অসম্ভব ভাল ছবি । এক কথায় অনবদ্য । বহুদিন পরে ...,1,0,0.973051,0.026949,0.973051
2380,এই নিয়া একটি সিনেমা বানানো উচিত । মানুষ ভাইরা...,1,0,0.972829,0.027171,0.972829
1107,ফিলিস্তিন ইস্যুতে প্রায় সব শ্রেনীর চাপাবাজেরা ...,1,0,0.972786,0.027214,0.972786
425,বাহিরে চলাফেরা করার পরও যদি মানুষের করোনা না হ...,1,0,0.972603,0.027397,0.972603


Top cross-dataset errors — FGM


,text,y_true,y_pred,prob_0,prob_1,pred_conf
2325,যেমন কর্ম তেমন ফল এখানে তার জন্য মায়া কান্দন ক...,1,0,0.973825,0.026175,0.973825
868,আপনার দৌড় কতটুকু জানা আছে । এই লাইন গুলোর কোন...,1,0,0.966306,0.033694,0.966306
2205,ব্যবসায়ীরা যতদিন ব্যবসা ছেড়ে রাজনীতি আর সরকারি...,1,0,0.964987,0.035013,0.964987
991,অসম্ভব ভাল ছবি । এক কথায় অনবদ্য । বহুদিন পরে ...,1,0,0.964614,0.035386,0.964614
2187,সব ফিমেইল স্টারদের ফেসবুকে ফলো করে তাদের ছবি দ...,1,0,0.963294,0.036706,0.963294
1288,লেভেল আর মেন্টালিটি নিয়ে কথা বলা মানুষগুলাই ক্...,1,0,0.963006,0.036994,0.963006
2380,এই নিয়া একটি সিনেমা বানানো উচিত । মানুষ ভাইরা...,1,0,0.953997,0.046003,0.953997
1123,জবার বয়স হয়েছে এবার রিটায়ারমেন্টের প্রয়োজন...,1,0,0.951456,0.048544,0.951456
2210,ইয়ার্কির একটা সীমা থাকা উচিৎ,1,0,0.950388,0.049612,0.950388
1648,বুঝলাম না পরী হাফা কি সাংবাদিকের চাকরি পাইছেন ...,1,0,0.949561,0.050439,0.949561


In [6]:
improved_cases, degraded_cases = compare_models(cross_plain, cross_fgm, n=20)

print("Cross-dataset cases fixed by FGM")
display(improved_cases)

print("Cross-dataset cases worsened by FGM")
display(degraded_cases)

Cross-dataset cases fixed by FGM


,text,y_true,base_pred,base_correct,base_conf,improved_pred,improved_correct,improved_conf
35,মোসাদ্দেক দলে থাকলে বাল ছিরা দিতো মফিজ সাংবাদি...,1,0,False,0.894791,1,True,0.590657
38,রিডেক্স শাস্তি প্রাপ্য । বর্তমানে তারা গ্রাহক ...,0,1,False,0.769817,0,True,0.501961
46,প্রভাবশালী প্রভাবশালী শব্দটা বাংলাদেশে শুনতে শ...,1,0,False,0.765251,1,True,0.593342
60,আপনি ঠিক জায়গায় আসছেন এখন নিজে ভর্তি হয়ে যান আ...,1,0,False,0.520472,1,True,0.567855
67,হয়তো বা ক্যারিয়ারের,0,1,False,0.714661,0,True,0.592338
84,বের না হলে জানতামই না যে দেশে এত পাগল,1,0,False,0.754465,1,True,0.614622
91,এখানে একদল চলে এসেছে কান্নার রিয়াক্ট দিচ্ছে ।...,1,0,False,0.829727,1,True,0.549088
107,মূর্খের শেষ ধাপ অতিক্রম করেছে যোগী বাবু উনাকে ...,1,0,False,0.768715,1,True,0.544548
114,পোস্ট কারী কে কেও ভুল বুঝবেন না প্লিজ ! গালিও ...,1,0,False,0.621315,1,True,0.568651
123,যাদের এখন গায়ে লাগবে তাদের দ্বারা সেভ না এটা শ...,0,1,False,0.684330,0,True,0.536286


Cross-dataset cases worsened by FGM


,text,y_true,base_pred,base_correct,base_conf,improved_pred,improved_correct,improved_conf
18,ওকে প্রথম ম্যাচে নিলে অবশ্যই বাংলাদেশ জিততে পারতো,0,0,True,0.770840,1,False,0.675388
33,সৌম্যকে নিয়ে আশাবাদী কে ?,0,0,True,0.789585,1,False,0.669634
39,আমি একজন অভিভাবক,0,0,True,0.589847,1,False,0.632106
48,এসআই রাফির বিরুদ্ধে চাঁদাবাজির অভিযোগ,0,0,True,0.701752,1,False,0.688300
55,লক ডাউন পুরোপুরি উঠিয়ে নিলে এই অবস্থা হতো না ।,0,0,True,0.891179,1,False,0.537253
58,অথচ প্রথম আলোই আবার প্রচার করেছে ইংল্যান্ডের চ...,0,0,True,0.573987,1,False,0.609291
68,সবই আলোচনায় থাকার ধান্দা । পুরাই কাদের মির্জা ।,1,1,True,0.556137,0,False,0.529444
69,তারা আজ বেঁচে থাকলে কিছু বলতেন ।,0,0,True,0.564569,1,False,0.791248
72,ইন্নালিল্লাহি ওয়া ইন্না ইলাইহি রাজিউন তাহলে এ...,0,0,True,0.546061,1,False,0.605834
74,কিছুক্ষণের মধ্যেই বিশিষ্ট কমেন্ট শিল্পী আপনাদে...,1,1,True,0.832901,0,False,0.513400


In [7]:
summary = pd.DataFrame([
    {"model": "cross_banglabert", "errors": int((~cross_plain["correct"]).sum())},
    {"model": "cross_banglabert_fgm", "errors": int((~cross_fgm["correct"]).sum())},
])
summary["error_reduction_vs_plain"] = summary.loc[0, "errors"] - summary["errors"]
display(summary)
summary.to_csv(TABLES / "cross_dataset_error_analysis_summary.csv", index=False)
print("Saved:", TABLES / "cross_dataset_error_analysis_summary.csv")

,model,errors,error_reduction_vs_plain
0,cross_banglabert,807,0
1,cross_banglabert_fgm,874,-67


Saved: ../04_outputs/tables/cross_dataset_error_analysis_summary.csv
